# YJMob observed mobility: motif diversity and regularity

This notebook classifies every classifiable observed user-day with fastmob at H3 resolution 8. It intentionally omits motif-prevalence plots and focuses on individual motif diversity and weekday/weekend regularity.

In [1]:
from pathlib import Path
import sys

import seaborn as sns
from IPython.display import display

PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / 'pyproject.toml').exists()
)
for path in [PROJECT_ROOT, PROJECT_ROOT / 'notebooks']:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from motif_analysis_utils import (
    classify_observed_daily_motifs, motif_diversity_and_regularity,
    plot_motif_diversity, plot_weekday_weekend_regularity,
)

sns.set_theme(style='whitegrid', context='notebook')
OBSERVED_PATH = PROJECT_ROOT / 'data' / 'yjmob' / 'yjmob_wgs84_simple.parquet'
assert OBSERVED_PATH.exists(), OBSERVED_PATH
print(f'Observed input: {OBSERVED_PATH}')

Observed input: /home/gustavo/citybehavex/data/yjmob/yjmob_wgs84_simple.parquet


In [2]:
analysis = classify_observed_daily_motifs(
    OBSERVED_PATH, uid_col='uid', timestamp_col='timestamp', location_col=None,
    lat_col='lat', lng_col='lon', h3_resolution=8,
)
daily = analysis['daily']
display(analysis['coverage'])
print(analysis['purpose_heuristic_warning'])
if not analysis['skipped_user_days'].empty:
    display(analysis['skipped_user_days'])

TypeError: compute_daily_motifs() takes 8 positional arguments but 9 were given

In [ ]:
summaries = motif_diversity_and_regularity(daily)
plot_motif_diversity(summaries['diversity'], 'YJMob: distinct daily motifs used per person')
display(summaries['diversity'].groupby('day_group', observed=True)['distinct_motifs'].describe().round(2))

In [ ]:
plot_weekday_weekend_regularity(
    summaries['regularity_long'], summaries['paired_regularity'], 'YJMob'
)
paired = summaries['paired_regularity']
print(f'People eligible for paired comparison: {len(paired):,}')
print(f'Mean weekday regularity: {paired["Weekday"].mean():.1%}')
print(f'Mean weekend regularity: {paired["Weekend"].mean():.1%}')
print(f'Mean weekend minus weekday difference: {(paired["Weekend"] - paired["Weekday"]).mean():+.1%}')

## Reading the results

Locations are spatially discretized at H3 resolution 8 before motif discovery. Diversity is the number of canonical daily graphs seen for a person. Regularity is the share of days matching their dominant motif, restricted to people with at least two observed days of each type; points above the diagonal are more regular on weekends.